# Mô hình CNN nhận diện biển báo – huấn luyện trên bộ dữ liệu biển báo **Việt Nam**
Bộ dữ liệu: [maitam/vietnamese-traffic-signs](https://www.kaggle.com/datasets/maitam/vietnamese-traffic-signs) (thay cho GTSRB).
Kiến trúc CNN **giữ nguyên** như `traffic_sign.py` (ảnh 30×30, thang 0–255, Conv32-32-64-64, Dense 256); chỉ lớp cuối đổi số đầu ra bằng số loại biển của bộ dữ liệu mới.

Notebook **tự nhận dạng** kiểu bộ dữ liệu (thư mục theo loại biển, hoặc ảnh có khoanh vùng YOLO / COCO / VOC / CSV) – nếu là ảnh khoanh vùng thì tự cắt biển báo ra.

**Cách chạy:** Runtime → Change runtime type → **T4 GPU** → **Runtime → Run all**. Lần đầu Kaggle có thể hỏi đăng nhập – làm theo hướng dẫn trên màn hình.

**Đầu ra:** `assets_app.zip` (gồm `model.tflite`, `signs.json`, thư mục `signs/`) → giải nén **đè** vào thư mục `assets/` của app (xóa các ảnh `signs/*.png` cũ trước) → đẩy lên GitHub → build lại.

Nếu ô 3 báo lỗi hoặc số loại biển trông không đúng, chụp màn hình kết quả **ô 2 và ô 3** gửi lại.

In [ ]:
# ===== Ô 1: Cài thư viện, kiểm tra GPU =====
!pip install -q kagglehub
import tensorflow as tf
print('TensorFlow', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ===== Ô 2: Tải bộ dữ liệu biển báo Việt Nam và xem cấu trúc =====
import os, glob, json, time, random, re, zipfile, shutil
import numpy as np, pandas as pd
from PIL import Image

DATASET = 'maitam/vietnamese-traffic-signs'
DATA_DIR = None   # nếu đã tải về Drive: DATA_DIR = '/content/drive/MyDrive/vietnamese-traffic-signs'
if DATA_DIR is None:
    import kagglehub
    DATA_DIR = kagglehub.dataset_download(DATASET)
print('Thư mục dữ liệu:', DATA_DIR)
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
for d, sub, files in list(os.walk(DATA_DIR))[:15]:
    so_anh = sum(f.lower().endswith(IMG_EXT) for f in files)
    khac = [f for f in files if not f.lower().endswith(IMG_EXT)][:4]
    print(f'  {d.replace(DATA_DIR, "") or "/"} | {len(sub)} thư mục con | {so_anh} ảnh | {khac}')

In [ ]:
# ===== Ô 3: Đọc ảnh và nhãn (tự nhận dạng kiểu bộ dữ liệu) =====
CHIA = {'train': 'train', 'training': 'train', 'val': 'val', 'valid': 'val', 'validation': 'val', 'test': 'test', 'testing': 'test'}
BO_QUA = {'images', 'image', 'imgs', 'data', 'dataset', 'labels'} | set(CHIA)

def phan_chia(path):
    for p in path.replace(DATA_DIR, '').lower().split(os.sep):
        if p in CHIA: return CHIA[p]
    return ''

def chuan(s):
    return re.sub(r'\s+', ' ', re.sub(r'[_]+', ' ', str(s).strip()))

mau = []   # (đường dẫn ảnh, tên lớp, hộp (x1,y1,x2,y2) hoặc None, phần chia)
ds_anh = [p for p in glob.glob(DATA_DIR + '/**/*', recursive=True) if p.lower().endswith(IMG_EXT)]
anh = {os.path.splitext(os.path.basename(p))[0]: p for p in ds_anh}

def tim_anh_yolo(t):   # labels/abc.txt -> images/abc.jpg (cùng phần train/val/test)
    goc = os.path.splitext(t.replace(os.sep + 'labels' + os.sep, os.sep + 'images' + os.sep))[0]
    for e in IMG_EXT + tuple(x.upper() for x in IMG_EXT):
        if os.path.exists(goc + e): return goc + e
    return anh.get(os.path.splitext(os.path.basename(t))[0])

# a) Kiểu YOLO: .txt cùng tên ảnh + data.yaml / classes.txt
names = None
for y in glob.glob(DATA_DIR + '/**/*.yaml', recursive=True):
    import yaml
    dy = yaml.safe_load(open(y))
    if isinstance(dy, dict) and 'names' in dy:
        names = dy['names'] if isinstance(dy['names'], list) else [dy['names'][k] for k in sorted(dy['names'])]
if names is None:
    for c in glob.glob(DATA_DIR + '/**/classes.txt', recursive=True) + glob.glob(DATA_DIR + '/**/*.names', recursive=True):
        names = [l.strip() for l in open(c) if l.strip()]
if names:
    for t in glob.glob(DATA_DIR + '/**/*.txt', recursive=True):
        f = tim_anh_yolo(t)
        if f is None or os.path.basename(t) == 'classes.txt': continue
        W, H = Image.open(f).size
        for line in open(t):
            v = line.split()
            if len(v) < 5: continue
            c, cx, cy, w, h = int(float(v[0])), *map(float, v[1:5])
            mau.append((f, names[c], ((cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H), phan_chia(t)))
    KIEU = 'YOLO (khoanh vùng)'
# b) Kiểu COCO (.json có annotations + categories)
if not mau:
    for j in glob.glob(DATA_DIR + '/**/*.json', recursive=True):
        try: dj = json.load(open(j))
        except Exception: continue
        if not (isinstance(dj, dict) and 'annotations' in dj and 'categories' in dj): continue
        cat = {c['id']: c['name'] for c in dj['categories']}
        imgs = {i['id']: i['file_name'] for i in dj['images']}
        for a in dj['annotations']:
            key = os.path.splitext(os.path.basename(imgs[a['image_id']]))[0]
            if key in anh:
                x, y, w, h = a['bbox']
                mau.append((anh[key], cat[a['category_id']], (x, y, x + w, y + h), phan_chia(j)))
    KIEU = 'COCO (khoanh vùng)'
# c) Kiểu Pascal VOC (.xml)
if not mau:
    import xml.etree.ElementTree as ET
    for xf in glob.glob(DATA_DIR + '/**/*.xml', recursive=True):
        try: r = ET.parse(xf).getroot()
        except Exception: continue
        key = os.path.splitext(os.path.basename(r.findtext('filename') or xf))[0]
        if key not in anh: continue
        for o in r.iter('object'):
            b = o.find('bndbox')
            mau.append((anh[key], o.findtext('name'), tuple(float(b.findtext(t)) for t in ('xmin', 'ymin', 'xmax', 'ymax')), phan_chia(xf)))
    KIEU = 'Pascal VOC (khoanh vùng)'
# d) Kiểu CSV: cột đường dẫn ảnh + cột nhãn (có thể kèm tọa độ)
if not mau:
    for cf in glob.glob(DATA_DIR + '/**/*.csv', recursive=True):
        t = pd.read_csv(cf); cols = {c.lower(): c for c in t.columns}
        cp = next((cols[c] for c in cols if c in ('path', 'filename', 'file', 'image', 'image_path', 'img')), None)
        cl = next((cols[c] for c in cols if c in ('classid', 'class', 'label', 'class_name', 'category', 'name')), None)
        if not (cp and cl): continue
        bx = [next((cols[c] for c in cols if c in k), None) for k in (('x1', 'xmin', 'roi.x1'), ('y1', 'ymin', 'roi.y1'), ('x2', 'xmax', 'roi.x2'), ('y2', 'ymax', 'roi.y2'))]
        for r in t.itertuples(index=False):
            r = r._asdict() if hasattr(r, '_asdict') else dict(zip(t.columns, r))
            p = str(r[cp]); f = os.path.join(os.path.dirname(cf), p)
            f = f if os.path.exists(f) else anh.get(os.path.splitext(os.path.basename(p))[0])
            if f: mau.append((f, r[cl], tuple(float(r[b]) for b in bx) if all(bx) else None, phan_chia(cf)))
    KIEU = 'CSV'
# e) Kiểu phân loại: mỗi thư mục con là một loại biển
if not mau:
    for f in ds_anh:
        lop = os.path.basename(os.path.dirname(f))
        if lop.lower() in BO_QUA: continue
        mau.append((f, lop, None, phan_chia(f)))
    KIEU = 'thư mục theo loại biển (phân loại)'
assert mau, 'Không đọc được nhãn – chụp màn hình kết quả ô 2 gửi lại để sửa.'

df = pd.DataFrame(mau, columns=['anh', 'lop', 'hop', 'chia'])
df['lop'] = df['lop'].map(chuan)
print('Kiểu bộ dữ liệu:', KIEU, '| số mẫu:', len(df), '| số loại biển:', df['lop'].nunique())
print(df['lop'].value_counts().to_string())

Ô 4: loại biển ít hơn `MIN_ANH` ảnh bị bỏ. Nếu bộ dữ liệu không chia sẵn train/test thì chia 70/15/15 **theo ảnh gốc** để các biển trong cùng một ảnh không lọt sang tập kiểm tra.

In [ ]:
# ===== Ô 4: Chọn nhãn, chia train / kiểm định / kiểm tra, cắt ảnh 30×30 =====
MIN_ANH = 10                                     # loại biển có ít hơn ngần này ảnh thì bỏ
dem = df['lop'].value_counts()
LOP = sorted(dem[dem >= MIN_ANH].index)
print('Bỏ', len(dem) - len(LOP), 'loại biển quá ít ảnh:', list(dem[dem < MIN_ANH].index))
df = df[df['lop'].isin(LOP)].copy()
df['nhan'] = df['lop'].map({t: i for i, t in enumerate(LOP)})
SO_LOP = len(LOP)
assert SO_LOP >= 2, 'Cần ít nhất 2 loại biển – kiểm tra lại kết quả ô 3.'

# Nếu bộ dữ liệu không chia sẵn: chia 70/15/15 theo ẢNH GỐC (mọi biển trong cùng một ảnh nằm chung một phần)
if (df['chia'] == 'test').sum() == 0:
    goc = df.drop_duplicates('anh')[['anh', 'nhan']]
    from sklearn.model_selection import train_test_split
    tr, te = train_test_split(goc, test_size=0.15, random_state=42, stratify=goc['nhan'] if goc['nhan'].value_counts().min() >= 2 else None)
    df['chia'] = np.where(df['anh'].isin(set(te['anh'])), 'test', 'train')
if (df['chia'] == 'val').sum() == 0:
    tr_idx = df[df['chia'] == 'train'].sample(frac=0.15 / 0.85, random_state=42).index
    df.loc[tr_idx, 'chia'] = 'val'
print(df.groupby('chia').size().to_string())

def cat_anh(r, le=0.1):
    im = Image.open(r.anh).convert('RGB')
    if r.hop is not None and not (isinstance(r.hop, float)):
        x1, y1, x2, y2 = r.hop; w, h = x2 - x1, y2 - y1
        im = im.crop((max(0, x1 - le * w), max(0, y1 - le * h), min(im.width, x2 + le * w), min(im.height, y2 + le * h)))
    return im

X, y, chia, ok = [], [], [], []
for r in df.itertuples():
    try:
        im = cat_anh(r)
        if min(im.size) < 8: continue
        X.append(np.array(im.resize((30, 30)))); y.append(r.nhan); chia.append(r.chia)
    except Exception as e:
        print('Bỏ qua', r.anh, e)
X, y, chia = np.array(X, np.uint8), np.array(y), np.array(chia)
X_train, y_train = X[chia == 'train'], y[chia == 'train']
X_val, y_val = X[chia == 'val'], y[chia == 'val']
X_test, y_test = X[chia == 'test'], y[chia == 'test']
print(f'Số lớp: {SO_LOP} | huấn luyện {len(X_train)} | kiểm định {len(X_val)} | kiểm tra {len(X_test)}')

# Ảnh minh họa cho app (assets/signs/<id>.png): ảnh rõ nhất (lớn nhất) của mỗi lớp
os.makedirs('signs', exist_ok=True)
for i, t in enumerate(LOP):
    g = df[df['nhan'] == i]
    best = max(g.head(30).itertuples(), key=lambda r: min(cat_anh(r).size))
    cat_anh(best).resize((128, 128)).save(f'signs/{i}.png')

In [ ]:
# ===== Ô 5: Tăng cường dữ liệu – giả lập ảnh chụp bằng điện thoại =====
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt

hinh_hoc = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.04, fill_mode='nearest'),          # nghiêng ±15°
    tf.keras.layers.RandomZoom((-0.25, 0.2), fill_mode='nearest'),      # cắt rộng/hẹp hơn
    tf.keras.layers.RandomTranslation(0.12, 0.12, fill_mode='nearest'), # biển lệch khỏi tâm khung cắt
])
K_MO = tf.constant(np.ones((3, 3, 3, 1), np.float32) / 9.0)

def bien_doi(x, y):
    x = tf.cast(x, tf.float32)
    x = hinh_hoc(x, training=True)
    n = tf.shape(x)[0]
    x = x * tf.random.uniform([n, 1, 1, 1], 0.5, 1.6) + tf.random.uniform([n, 1, 1, 1], -40.0, 40.0)
    tb = tf.reduce_mean(x, axis=[1, 2, 3], keepdims=True)
    x = (x - tb) * tf.random.uniform([n, 1, 1, 1], 0.7, 1.4) + tb
    xam = tf.reduce_mean(x, axis=3, keepdims=True)
    x = xam + (x - xam) * tf.random.uniform([n, 1, 1, 1], 0.6, 1.5)
    mo = tf.nn.depthwise_conv2d(x, K_MO, [1, 1, 1, 1], 'SAME')
    x = tf.where(tf.random.uniform([n, 1, 1, 1]) < 0.3, mo, x)
    x = x + tf.random.normal(tf.shape(x), 0.0, 1.0) * tf.random.uniform([n, 1, 1, 1], 0.0, 8.0)
    return tf.clip_by_value(x, 0.0, 255.0), y   # giữ thang 0–255 như traffic_sign.py

BATCH = 32
ds_train = (tf.data.Dataset.from_tensor_slices((X_train, to_categorical(y_train, SO_LOP)))
            .shuffle(len(X_train), seed=42).batch(BATCH).map(bien_doi, num_parallel_calls=tf.data.AUTOTUNE)
            .prefetch(tf.data.AUTOTUNE))
ds_val = tf.data.Dataset.from_tensor_slices((X_val.astype(np.float32), to_categorical(y_val, SO_LOP))).batch(256)

xb, _ = next(iter(ds_train))
plt.figure(figsize=(8, 2.2))
for k in range(min(16, len(xb))):
    plt.subplot(2, 8, k + 1); plt.imshow(xb[k].numpy().astype('uint8')); plt.axis('off')
plt.suptitle('Ảnh huấn luyện sau tăng cường'); plt.savefig('anh_tang_cuong.png', dpi=150); plt.show()

In [ ]:
# ===== Ô 6: Huấn luyện mô hình CNN của nhóm (kiến trúc giữ nguyên, chỉ đổi số lớp đầu ra) =====
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Conv2D, MaxPool2D, Dense, Flatten, Dropout

model = Sequential()
model.add(Conv2D(filters=32, padding='same', kernel_size=(5, 5), activation='relu', input_shape=(30, 30, 3)))
model.add(Conv2D(filters=32, padding='same', kernel_size=(5, 5), activation='relu'))
model.add(MaxPool2D(pool_size=(2, 2)))
model.add(Dropout(rate=0.25))
model.add(Conv2D(filters=64, padding='same', kernel_size=(3, 3), activation='relu'))
model.add(Conv2D(filters=64, padding='same', kernel_size=(3, 3), activation='relu'))
model.add(MaxPool2D(pool_size=(2, 2)))
model.add(Dropout(rate=0.25))
model.add(Flatten())
model.add(Dense(256, activation='relu'))
model.add(Dropout(rate=0.5))
model.add(Dense(SO_LOP, activation='softmax'))
model.summary()

# Lớp ít ảnh được tăng trọng số để mô hình không chỉ học các lớp nhiều ảnh
from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.arange(SO_LOP), y=y_train) if len(set(y_train)) == SO_LOP else None
epochs = 30
model.compile(loss='categorical_crossentropy', optimizer=tf.keras.optimizers.Adam(1e-3), metrics=['accuracy'])
goi_lai = [tf.keras.callbacks.ModelCheckpoint('tot_nhat.keras', monitor='val_accuracy', save_best_only=True),
           tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1)]
t0 = time.time()
history = model.fit(ds_train, epochs=epochs, validation_data=ds_val, callbacks=goi_lai,
                    class_weight=None if cw is None else dict(enumerate(cw)))
train_min = (time.time() - t0) / 60
model = load_model('tot_nhat.keras')
model.save('traffic_classifier_vn.h5')

ep = range(1, epochs + 1)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].plot(ep, history.history['accuracy'], 'o-', label='Huấn luyện'); ax[0].plot(ep, history.history['val_accuracy'], 's-', label='Kiểm định')
ax[0].set_xlabel('Epoch'); ax[0].set_title('(a) Độ chính xác'); ax[0].grid(alpha=.3); ax[0].legend()
ax[1].plot(ep, history.history['loss'], 'o-', label='Huấn luyện'); ax[1].plot(ep, history.history['val_loss'], 's-', label='Kiểm định')
ax[1].set_xlabel('Epoch'); ax[1].set_title('(b) Hàm mất mát'); ax[1].grid(alpha=.3); ax[1].legend()
plt.tight_layout(); plt.savefig('duong_cong_huan_luyen_vn.png', dpi=200); plt.show()

In [ ]:
# ===== Ô 7: Đánh giá trên tập kiểm tra (Accuracy, Precision, Recall, F1 từng lớp) =====
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
pt = model.predict(X_test.astype(np.float32), verbose=0)
p_test, c_test = pt.argmax(1), pt.max(1)
acc = accuracy_score(y_test, p_test)
P, R, F, S = precision_recall_fscore_support(y_test, p_test, labels=list(range(SO_LOP)), zero_division=0)
bang = pd.DataFrame({'Nhãn': range(SO_LOP), 'Loại biển': LOP, 'Số ảnh test': S,
                     'Precision': P.round(2), 'Recall': R.round(2), 'F1': F.round(2)})
bang.to_csv('bang_precision_recall_f1_vn.csv', index=False)
print(bang.to_string(index=False))
cm = confusion_matrix(y_test, p_test, labels=list(range(SO_LOP))); np.fill_diagonal(cm, 0)
nham = sorted(((cm[i, j], LOP[i], LOP[j]) for i in range(SO_LOP) for j in range(SO_LOP) if cm[i, j]), reverse=True)[:10]
print('\nCác cặp hay nhầm nhất (số lần, biển thật → đoán thành):')
for k in nham: print(' ', k)
nguong = c_test >= 0.6
print(f'\nNgưỡng 60%: {nguong.mean() * 100:.1f}% ảnh vượt ngưỡng, trong đó đúng {accuracy_score(y_test[nguong], p_test[nguong]) * 100:.2f}%')

Ô 8: tên hiển thị trong app. Nếu tên lớp của bộ dữ liệu là tiếng Anh hoặc mã số, điền tên tiếng Việt vào `TEN` rồi chạy lại ô 8 và ô 9 (không cần huấn luyện lại).

In [ ]:
# ===== Ô 8: Tên tiếng Việt cho app (signs.json) =====
# Điền tên/mã hiệu/ý nghĩa theo QCVN 41:2024/BGTVT cho từng loại biển rồi CHẠY LẠI Ô 8 và Ô 9.
# Loại nào chưa điền sẽ dùng tên thư mục gốc của bộ dữ liệu.
TEN = {
    # 'tên lớp trong bộ dữ liệu': ('Tên tiếng Việt', 'Mã hiệu', 'Nhóm biển', 'Ý nghĩa'),
    # 'no parking': ('Cấm đỗ xe', 'P.131a', 'Biển báo cấm', 'Cấm các loại xe đỗ ở phía đường có đặt biển.'),
}
def nhom(ma):
    return {'P': 'Biển báo cấm', 'R': 'Biển hiệu lệnh', 'W': 'Biển báo nguy hiểm', 'I': 'Biển chỉ dẫn', 'S': 'Biển phụ'}.get(str(ma)[:1].upper(), '')
signs = []
for i, t in enumerate(LOP):
    m = re.match(r'^([PRWIS])[\.\s_-]?(\d+[a-z]?)', t, re.I)       # tên thư mục kiểu "P.102", "W_207a"…
    ma = f'{m.group(1).upper()}.{m.group(2)}' if m else ''
    ten, ma2, nh, yn = TEN.get(t, (t, ma, nhom(ma) or 'Khác', 'Chưa có mô tả.'))
    signs.append({'id': i, 'name': ten, 'code': ma2, 'group': nh, 'meaning': yn})
json.dump(signs, open('signs.json', 'w'), ensure_ascii=False, indent=1)
print(pd.DataFrame(signs).to_string(index=False))

In [ ]:
# ===== Ô 9: Xuất model.tflite (lô = 1, số thực 32 bit) và tải về =====
m1 = Sequential()
m1.add(tf.keras.layers.Input((30, 30, 3), batch_size=1))
for layer in model.layers:
    m1.add(layer.__class__.from_config(layer.get_config()))
m1.set_weights(model.get_weights())
open('model.tflite', 'wb').write(tf.lite.TFLiteConverter.from_keras_model(m1).convert())
it = tf.lite.Interpreter(model_path='model.tflite'); it.allocate_tensors()
it.set_tensor(it.get_input_details()[0]['index'], X_test[:1].astype(np.float32)); it.invoke()
sai_khac = np.abs(it.get_tensor(it.get_output_details()[0]['index']) - pt[:1]).max()

with zipfile.ZipFile('assets_app.zip', 'w') as z:      # giải nén đè vào thư mục assets/ của app
    z.write('model.tflite', 'model.tflite'); z.write('signs.json', 'signs.json')
    for f in sorted(glob.glob('signs/*.png')): z.write(f, f)

print('\n================ SỐ LIỆU ================')
print(f'Bộ dữ liệu: {DATASET} ({KIEU}) | {SO_LOP} loại biển | huấn luyện/kiểm định/kiểm tra: {len(X_train)}/{len(X_val)}/{len(X_test)}')
print(f'{epochs} epoch, batch {BATCH}, Adam 0.001 | thời gian huấn luyện: {train_min:.1f} phút | tham số: {model.count_params():,}')
print(f'Độ chính xác kiểm định tốt nhất: {max(history.history["val_accuracy"]) * 100:.2f}%')
print(f'Tập kiểm tra: {acc * 100:.2f}% ({int((p_test == y_test).sum())}/{len(y_test)}) | Macro P/R/F1: {P.mean():.2f} / {R.mean():.2f} / {F.mean():.2f}')
print(f'model.tflite: {os.path.getsize("model.tflite") / 2**20:.2f} MB | sai khác so với Keras: {sai_khac:.1e}')

try:
    from google.colab import files
    for f in ['assets_app.zip', 'traffic_classifier_vn.h5', 'bang_precision_recall_f1_vn.csv',
              'duong_cong_huan_luyen_vn.png', 'anh_tang_cuong.png']:
        files.download(f)
except ImportError:
    pass